# Goldstandard-Auswahl: 150 Sätze aus 20 trainingsfreien Märchen

## Schritt 0 — Konfiguration


In [19]:
import re, csv, random
from pathlib import Path

EINGABE = Path("../02_quelldaten/pool_maerchen")     # Ordner mit den 20 *.txt (Präfix 0_-3_)
AUSGABE = Path("../02_quelldaten")  # Zielordner der drei Ausgabedateien
SEED = 42

SAMMLUNG = {"0": "Arabisch", "1": "Grimm", "2": "Andersen", "3": "Japanisch"}
QUOTE_38 = {"Arabisch", "Grimm"}          # erhalten 38 statt 37 Sätze
PARITAET_T1 = {"Arabisch": True, "Grimm": True, "Andersen": True,
               "Japanisch": False}        # ergibt exakt 75/75

ANFUEHRUNGSZEICHEN = {'"', "„", "“", "”", "»", "«", "›", "‹", "‚", "‘", "’"}
SATZSCHLUSSZEICHEN = {".", "!", "?", "…"}
print(f"Eingabe: {EINGABE} | Ausgabe: {AUSGABE} | Seed: {SEED}")

Eingabe: ../02_quelldaten/pool_maerchen | Ausgabe: ../02_quelldaten | Seed: 42


## Schritt 1 — Projekt-Bausteine der Segmentierung


In [4]:
def normalisiere_text(text):
    """Silbentrennung am Zeilenende aufheben, Umbrüche -> Leerzeichen."""
    text = re.sub(r"(\w)-\n([a-zäöüß])", r"\1\2", text)
    text = re.sub(r"\s*\n\s*", " ", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    return text.strip()


def quote_grenzen(doc):
    """Satzgrenzen an Anführungszeichen korrigieren (vor dem Parser)."""
    for i in range(1, len(doc)):
        tok = doc[i]
        if tok.text not in ANFUEHRUNGSZEICHEN:
            continue
        haengt_links = doc[i - 1].whitespace_ == ""
        haengt_rechts = tok.whitespace_ == "" and i + 1 < len(doc)
        if haengt_links:
            tok.is_sent_start = False
        elif haengt_rechts and doc[i - 1].text in SATZSCHLUSSZEICHEN:
            tok.is_sent_start = True
            doc[i + 1].is_sent_start = False
    return doc


def repariere_satzspannen(spans):
    """Sicherheitsnetz: verirrte Anführungszeichen/leere Sätze reparieren."""
    spans = list(spans)
    repariert, i = [], 0
    while i < len(spans):
        span = spans[i]
        doc = span.doc
        while (len(span) > 0 and span[0].text in ANFUEHRUNGSZEICHEN
               and span.start > 0 and doc[span.start - 1].whitespace_ == ""
               and repariert):
            repariert[-1] = doc[repariert[-1].start: span.start + 1]
            span = doc[span.start + 1: span.end]
        if len(span) == 0:
            i += 1; continue
        if all(not t.is_alpha and not t.like_num for t in span):
            if span[-1].whitespace_ == "" and i + 1 < len(spans):
                spans[i + 1] = doc[span.start: spans[i + 1].end]
            elif repariert:
                repariert[-1] = doc[repariert[-1].start: span.end]
            else:
                repariert.append(span)
            i += 1; continue
        repariert.append(span); i += 1
    return repariert

print("Segmentierungs-Bausteine geladen.")

Segmentierungs-Bausteine geladen.


## Schritt 2 — Goldstandard-spezifische Textbereinigung


In [5]:
import spacy
from spacy.language import Language

def bereinige_rohtext(roh):
    zeilen = roh.replace("\r\n", "\n").split("\n")
    zeilen = [z for z in zeilen
              if not re.match(r"^\s*\[(Abbildung|\d+)\]", z)]
    for k, z in enumerate(zeilen):                 # isolierte Überschriften
        vor_leer = k == 0 or not zeilen[k - 1].strip()
        nach_leer = k == len(zeilen) - 1 or not zeilen[k + 1].strip()
        w = z.strip().replace("_", "")
        if vor_leer and nach_leer and re.match(r"^[A-ZÄÖÜ][A-Za-zäöüß]*\.?$", w):
            zeilen[k] = ""
    for k, z in enumerate(zeilen):                 # Titelzeile
        if z.strip():
            w = z.strip()
            if len(w) <= 60 and len(w.split()) <= 7:
                zeilen[k] = ""
            break
    text = "\n".join(zeilen)
    text = re.sub(r"\[\d+\]", "", text)
    text = re.sub(r"_+", "", text)
    text = re.sub(r"\s*--\s*", " – ", text)
    return text


nlp = spacy.load("de_core_news_md",
                 disable=["lemmatizer", "attribute_ruler", "ner"])
if not Language.has_factory("quote_grenzen"):
    Language.component("quote_grenzen", func=quote_grenzen)
if "quote_grenzen" not in nlp.pipe_names:
    nlp.add_pipe("quote_grenzen", before="parser")
nlp.max_length = 400_000

def lade_saetze(pfad):
    text = normalisiere_text(bereinige_rohtext(pfad.read_text("utf-8")))
    return repariere_satzspannen(nlp(text).sents)

print("Bereinigung und Pipeline bereit:", nlp.pipe_names)

/home/timc/Workspace/fairytale-personality-profiling/venv311_fairytale-profiling/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Bereinigung und Pipeline bereit: ['tok2vec', 'tagger', 'morphologizer', 'quote_grenzen', 'parser']


## Schritt 3 — Modellunabhängige Figuren-Heuristik



In [6]:
FIGUR_LEXIKON = {
    "mann","männer","manne","frau","frauen","weib","weiber","mädchen",
    "kind","kinder","knabe","knaben","junge","jüngling","jünglinge",
    "jungfrau","jungfern","greis","alte","alter","alten","mutter",
    "mütterchen","vater","sohn","söhne","tochter","töchter","bruder",
    "brüder","schwester","schwestern","gemahl","gemahlin","gatte","gattin",
    "braut","bräutigam","witwe","base","basen","tante","tanten","vetter",
    "oheim","nachbar","nachbarin","nachbarssohn",
    "könig","königin","königssohn","königstochter","prinz","prinzessin",
    "kaiser","kaiserin","fürst","fürstin","herr","herrin","ritter",
    "soldat","soldaten","krieger","hauptmann","räuber","dieb","betrüger",
    "weber","müller","müllerin","fischer","jäger","hirt","hirtin",
    "gänsehirtin","schuster","schusterin","schneider","koch","köchin",
    "magd","diener","dienerin","dienerschar","knecht","bote","kaufmann",
    "handelsmann","kaufherr","wirt","wirtin","priester","pfarrer",
    "pfarrerin","geistliche","geistlicher","küster","kantor","minister",
    "staatsmann","staatsmänner","beamte","kammerherr","kammerherren",
    "scharfrichter","kutscher","spinnerin","spinnerinnen","pflegerin",
    "arbeiterin","bettelmädchen","landbesitzer",
    "hexe","zauberer","zauberin","fee","geist","geister","kobold",
    "kobolde","riese","zwerg","engel","teufel","drache","schlange",
    "riesenschlange","nixe","seejungfrau","meerkönig","sandmann",
    "gott","götter","göttin","held","mörder",
    "tanuki","oni","daimyo","daimio","mikado","eremit","dschinn",
    "geisterfürst","kalif","kalifen","sultan","sultanin","wesir","vezier",
    "großwesir","scheich","derwisch","sklave","sklavin","sklaven","mohr",
    "beduine",
    "hund","hündchen","affe","fasan","vogel","frosch","frösche",
}
PRONOMEN = {"er", "sie", "es", "ihm", "ihr", "ihn", "ihnen"}

def figuren_score(span):
    n = 0
    for t in span:
        if t.text.lower() in FIGUR_LEXIKON:
            n += 1
        elif t.pos_ == "PROPN" and t.is_alpha and len(t.text) > 2:
            n += 1
    return n

def pronomen_zahl(span):
    return sum(t.text.lower() in PRONOMEN for t in span)

print(f"Lexikon: {len(FIGUR_LEXIKON)} Rollenwörter.")

Lexikon: 166 Rollenwörter.


## Schritt 4 — Blockwahl innerhalb einer Erzählhälfte


In [7]:
def waehle_block(rng, saetze, lo, hi, laenge, mindest=2):
    scores = [figuren_score(s) for s in saetze]
    starts = list(range(lo, hi - laenge + 1))
    if not starts:
        raise ValueError("Erzählhälfte kürzer als Blocklänge")
    bester, bester_score = None, -1
    for _ in range(300):
        start = rng.choice(starts)
        blk = scores[start:start + laenge]
        treffer = sum(1 for x in blk if x > 0)
        if treffer >= min(mindest, laenge) and sum(blk) >= laenge - 1:
            return start
        if sum(blk) > bester_score:
            bester, bester_score = start, sum(blk)
    return bester

print("Blockwahl definiert.")

Blockwahl definiert.


## Schritt 5 — Erzählungen einlesen


In [16]:
erzaehlungen = []
for pfad in sorted(EINGABE.glob("*.txt")):
    if pfad == EINGABE / "_README.txt": # README-Datei überspringen
        print("README übersprungen..")
        continue
    praefix = pfad.stem.split("_", 1)[0]
    eid = re.sub(r"-mod$", "", pfad.stem)
    eid = re.sub(r"_{2,}", "_", eid)
    saetze = lade_saetze(pfad)
    erzaehlungen.append(dict(id=eid, sammlung=SAMMLUNG[praefix],
                             datei=pfad.name, saetze=saetze))
    print(f"  [{SAMMLUNG[praefix]:<9}] {eid[:52]:<52} {len(saetze):>4} Sätze")
assert len(erzaehlungen) == 20, "Pool unvollständig?"

  [Arabisch ] 0_Alaeddin und die Wunderlampe                       1467 Sätze
  [Arabisch ] 0_Die Abenteuer Sindbads des Seefahrers              1048 Sätze
  [Arabisch ] 0_Die Geschichte von Ali dem Kairenser und dem spuke  326 Sätze
  [Arabisch ] 0_Dschudar und seine Brüder                           796 Sätze
  [Arabisch ] 0_Khalifah der Fischer von Bagdad                     396 Sätze
  [Grimm    ] 1_Der Eisenhans                                       151 Sätze
  [Grimm    ] 1_Der Räuberbräutigam                                  64 Sätze
  [Grimm    ] 1_Die Gänsehirtin am Brunnen                          208 Sätze
  [Grimm    ] 1_Die drei Spinnerinnen                                36 Sätze
  [Grimm    ] 1_Märchen von einem, der auszog das Fürchten zu lern  169 Sätze
  [Andersen ] 2_Das Märchen vom Sandmann                            206 Sätze
  [Andersen ] 2_Der Engel                                            42 Sätze
  [Andersen ] 2_Des Kaisers neue Kleider                        

## Schritt 6 — Zuteilung und Blockziehung


In [17]:
rng = random.Random(SEED)
bloecke = []
for samm in ["Arabisch", "Grimm", "Andersen", "Japanisch"]:
    gruppe = sorted([e for e in erzaehlungen if e["sammlung"] == samm],
                    key=lambda e: -len(e["saetze"]))
    n_acht = 3 if samm in QUOTE_38 else 2
    for rang, erz in enumerate(gruppe):
        n = 8 if rang < n_acht else 7
        b1_len, b2_len = 4, n - 4
        N = len(erz["saetze"]); mitte = N // 2
        s1 = waehle_block(rng, erz["saetze"], 0, mitte, b1_len)
        s2 = waehle_block(rng, erz["saetze"],
                          max(mitte, s1 + b1_len + 2), N, b2_len)
        b1_t1 = (rang % 2 == 0) == PARITAET_T1[samm]
        for name, start, ln, t1 in [("A", s1, b1_len, b1_t1),
                                    ("B", s2, b2_len, not b1_t1)]:
            bloecke.append(dict(erz=erz, block=name, start=start,
                                laenge=ln, tranche=1 if t1 else 2,
                                spans=erz["saetze"][start:start + ln]))
print(f"{len(bloecke)} Blöcke gezogen | "
      f"Sätze: {sum(b['laenge'] for b in bloecke)}")

40 Blöcke gezogen | Sätze: 150


## Schritt 7 — Ausgaben schreiben und Summenstatistik


In [22]:
AUSGABE.mkdir(parents=True, exist_ok=True)
conll, ueb, stat = [], [], {}
kopf = ("# Goldstandard Version 1 — 150 Sätze aus 20 trainingsfreien "
        "Märchen\n# Auswahl: auswahl_goldstandard, Seed "
        f"{SEED}. NER-Spalte von Hand annotieren (O -> B-FIGUR/I-FIGUR).\n")
conll.append(kopf)
for b in sorted(bloecke, key=lambda x: (x["erz"]["sammlung"],
                                        x["erz"]["id"], x["block"])):
    e = b["erz"]
    conll += [f"# erzaehlung = {e['id']}", f"# sammlung = {e['sammlung']}",
              f"# tranche = {b['tranche']}", f"# block = {b['block']}",
              f"# satzindizes = {b['start']}-{b['start']+b['laenge']-1}"]
    for span in b["spans"]:
        for t in span:
            conll.append(f"{t.text}\t{t.pos_}\tO")   # UPOS wie Training
        conll.append("")
    s = stat.setdefault((e["sammlung"], e["id"]),
                        dict(saetze=0, t1=0, t2=0, fig=0, pron=0,
                             gesamt=len(e["saetze"])))
    s["saetze"] += b["laenge"]
    s["t1" if b["tranche"] == 1 else "t2"] += b["laenge"]
    s["fig"] += sum(figuren_score(x) for x in b["spans"])
    s["pron"] += sum(pronomen_zahl(x) for x in b["spans"])
    kontext = (e["saetze"][b["start"] - 1].text
               if b["start"] > 0 else "(Erzählungsanfang)")
    ueb.append(f"\n### {e['id']} — Block {b['block']} "
               f"(Tranche {b['tranche']}, Sätze {b['start']}–"
               f"{b['start']+b['laenge']-1} von {len(e['saetze'])})\n")
    ueb.append(f"*[Kontext, nicht annotieren]* {kontext}\n")
    for k, span in enumerate(b["spans"]):
        ueb.append(f"{b['start']+k}. {span.text}")

(AUSGABE / "goldstandard_v1.conll").write_text(
    "\n".join(conll) + "\n", encoding="utf-8")
with open(AUSGABE / "goldstandard_v1_meta.csv", "w", newline="",
          encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(["erzaehlung", "sammlung"])
    for e in erzaehlungen:
        w.writerow([e["id"], e["sammlung"]])

zeilen = ["# Goldstandard V1 — Auswahlübersicht\n"]
pro = {}
for (samm, eid), s in sorted(stat.items()):
    ps = pro.setdefault(samm, dict(saetze=0, t1=0, t2=0, fig=0, pron=0))
    for k in ps: ps[k] += s[k]
zeilen += ueb
(AUSGABE / "goldstandard_v1_uebersicht.md").write_text(
    "\n".join(zeilen) + "\n", encoding="utf-8")

ges = sum(p["saetze"] for p in pro.values())
t1 = sum(p["t1"] for p in pro.values())
print(f"Gesamt: {ges} Sätze | Tranche 1: {t1} | Tranche 2: {ges-t1}")
for samm, p in sorted(pro.items()):
    print(f"  {samm:<9} {p['saetze']} (T1 {p['t1']}/T2 {p['t2']}), "
          f"≈{p['fig']} Figuren, {p['pron']} Pronomen")

Gesamt: 150 Sätze | Tranche 1: 75 | Tranche 2: 75
  Andersen  37 (T1 19/T2 18), ≈43 Figuren, 47 Pronomen
  Arabisch  38 (T1 19/T2 19), ≈41 Figuren, 61 Pronomen
  Grimm     38 (T1 19/T2 19), ≈47 Figuren, 60 Pronomen
  Japanisch 37 (T1 18/T2 19), ≈40 Figuren, 59 Pronomen
